# Data Science & Database Assignment: Session 6
## Topic: ORDER BY + Sorting

**Student Name:** Atharv Chaudhari  
**Course:** Data Science & Database Fundamentals  
**Database Name:** `sorting_app.db`  
**Core Concepts:** Single-Column Sorting (`ASC` / `DESC`), Top-N Filtering (`LIMIT`), Multi-Column Hierarchical Sorting, Tie-Breaking Logic  
**Status:** Completed & Pre-Executed  

---

### Logical Order of Sorting in SQL

In SQL Query Processing, `ORDER BY` is executed **almost last** in the pipeline (just before `LIMIT`):

```text
FROM -> WHERE -> GROUP BY -> HAVING -> SELECT -> DISTINCT -> ORDER BY -> LIMIT
```

Key sorting rules:
* **`ASC` (Default):** Ascending order (lowest to highest, A to Z, oldest date to newest date).
* **`DESC`:** Descending order (highest to lowest, Z to A, newest date to oldest date).
* **Multi-Column:** When the first column contains identical values (ties), the SQL engine falls back to the secondary column to resolve the ordering.


## Task 1: Sort Products by Price Ascending (Flipkart Style)
> **Problem Statement:** Write an SQL query to display all products from a `'products'` table and sort them by price in ascending order, similar to how Flipkart lists items from lowest to highest price.

### SQL Query:
```sql
SELECT * 
FROM products
ORDER BY price ASC;
```
*(Note: `ASC` is optional since ascending is the SQL default, but explicitly specifying `ASC` improves code readability and intent).*


In [1]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("sorting_app.db")
cursor = conn.cursor()

# Create products table
cursor.execute("DROP TABLE IF EXISTS products;")
cursor.execute("""
CREATE TABLE products (
    id INTEGER PRIMARY KEY,
    product_name VARCHAR(255) NOT NULL,
    category VARCHAR(100) NOT NULL,
    price DECIMAL(10, 2) NOT NULL
);
""")

# Insert 8 catalog products across diverse price points
products_data = [
    (1, 'Boat Rockerz 450 Headphone', 'Electronics', 1499.00),
    (2, 'Apple iPhone 15', 'Mobiles', 71999.00),
    (3, 'Cotton Casual Shirt', 'Fashion', 799.00),
    (4, 'Samsung Galaxy S24', 'Mobiles', 79999.00),
    (5, 'Sony WH-1000XM5 Headset', 'Electronics', 28990.00),
    (6, 'Fastrack Analog Watch', 'Fashion', 1250.00),
    (7, 'MacBook Air M2', 'Laptops', 94990.00),
    (8, 'Logitech Wireless Mouse', 'Accessories', 599.00)
]
cursor.executemany("INSERT INTO products VALUES (?, ?, ?, ?);", products_data)
conn.commit()

# Execute Task 1 query
query_t1 = "SELECT * FROM products ORDER BY price ASC;"
df_t1 = pd.read_sql_query(query_t1, conn)
print("Task 1 Output: Products Sorted by Price (Lowest to Highest - Flipkart Style)")
display(df_t1)


Task 1 Output: Products Sorted by Price (Lowest to Highest - Flipkart Style)


,id,product_name,category,price
0,8,Logitech Wireless Mouse,Accessories,599
1,3,Cotton Casual Shirt,Fashion,799
2,6,Fastrack Analog Watch,Fashion,1250
3,1,Boat Rockerz 450 Headphone,Electronics,1499
4,5,Sony WH-1000XM5 Headset,Electronics,28990
5,2,Apple iPhone 15,Mobiles,71999
6,4,Samsung Galaxy S24,Mobiles,79999
7,7,MacBook Air M2,Laptops,94990


## Task 2: Top 5 Most Expensive Products (`ORDER BY DESC LIMIT 5`)
> **Problem Statement:** Modify your previous query to show the top 5 most expensive products using `ORDER BY` with `DESC` and `LIMIT`.

### SQL Query:
```sql
SELECT * 
FROM products
ORDER BY price DESC
LIMIT 5;
```
* **`ORDER BY price DESC`**: Reverses the order so the highest priced items appear first.
* **`LIMIT 5`**: Truncates the result set to return only the first 5 records.


In [2]:
query_t2 = """
SELECT * 
FROM products
ORDER BY price DESC
LIMIT 5;
"""

df_t2 = pd.read_sql_query(query_t2, conn)
print("Task 2 Output: Top 5 Most Expensive Products")
display(df_t2)


Task 2 Output: Top 5 Most Expensive Products


,id,product_name,category,price
0,7,MacBook Air M2,Laptops,94990
1,4,Samsung Galaxy S24,Mobiles,79999
2,2,Apple iPhone 15,Mobiles,71999
3,5,Sony WH-1000XM5 Headset,Electronics,28990
4,1,Boat Rockerz 450 Headphone,Electronics,1499


## Task 3: Multi-Column Hierarchical Sort (`release_year DESC, rating DESC`)
> **Problem Statement:** Given a `'movies'` table with columns `'title'`, `'release_year'`, and `'rating'`, write an SQL query to list all movies sorted first by `release_year` in descending order (latest first), then by `rating` in descending order (highest rated first).

### SQL Query:
```sql
SELECT title, release_year, rating
FROM movies
ORDER BY release_year DESC, rating DESC;
```
* **Evaluation Order:**
  1. The database groups movies by `release_year` in reverse chronological order (2024 $ightarrow$ 2023 $ightarrow$ 2014 $ightarrow$ 2010).
  2. For movies released in the same year (e.g., 2024 or 2023), the engine sorts them internally by `rating` from highest to lowest.


In [3]:
# Create movies table
cursor.execute("DROP TABLE IF EXISTS movies;")
cursor.execute("""
CREATE TABLE movies (
    id INTEGER PRIMARY KEY,
    title VARCHAR(255) NOT NULL,
    release_year INTEGER NOT NULL,
    rating DECIMAL(3, 1) NOT NULL
);
""")

movies_data = [
    (1, 'Oppenheimer', 2023, 8.9),
    (2, 'Barbie', 2023, 7.3),
    (3, 'Dune: Part Two', 2024, 8.6),
    (4, 'Furiosa', 2024, 7.6),
    (5, 'Interstellar', 2014, 8.7),
    (6, 'Inception', 2010, 8.8),
    (7, 'Spider-Man: Across the Spider-Verse', 2023, 8.7)
]
cursor.executemany("INSERT INTO movies VALUES (?, ?, ?, ?);", movies_data)
conn.commit()

query_t3 = """
SELECT title, release_year, rating
FROM movies
ORDER BY release_year DESC, rating DESC;
"""

df_t3 = pd.read_sql_query(query_t3, conn)
print("Task 3 Output: Movies Sorted by Release Year (Latest) then Rating (Highest)")
display(df_t3)


Task 3 Output: Movies Sorted by Release Year (Latest) then Rating (Highest)


,title,release_year,rating
0,Dune: Part Two,2024,8.6
1,Furiosa,2024,7.6
2,Oppenheimer,2023,8.9
3,Spider-Man: Across the Spider-Verse,2023,8.7
4,Barbie,2023,7.3
5,Interstellar,2014,8.7
6,Inception,2010,8.8


## Task 4: Alphabetical Restaurant Listing (`A-Z` with `LIMIT 10`)
> **Problem Statement:** Write an SQL query to display the first 10 restaurants from a `'restaurants'` table, sorted alphabetically by name, just like Zomato's A-Z listing.  
> *Hint: Use `ORDER BY` with `LIMIT`.*

### SQL Query:
```sql
SELECT * 
FROM restaurants
ORDER BY name ASC
LIMIT 10;
```


In [4]:
# Create restaurants table
cursor.execute("DROP TABLE IF EXISTS restaurants;")
cursor.execute("""
CREATE TABLE restaurants (
    id INTEGER PRIMARY KEY,
    name VARCHAR(255) NOT NULL,
    cuisine VARCHAR(100) NOT NULL,
    rating DECIMAL(2, 1) NOT NULL,
    city VARCHAR(100) NOT NULL
);
""")

restaurants_data = [
    (1, 'Swagat Restaurant', 'South Indian', 4.3, 'Ahmedabad'),
    (2, 'Barbeque Nation', 'Buffet / BBQ', 4.4, 'Ahmedabad'),
    (3, 'Cafe De Bistro', 'Continental', 3.9, 'Surat'),
    (4, "Domino's Pizza", 'Italian / Fast Food', 4.1, 'Ahmedabad'),
    (5, "Haldiram's", 'North Indian / Sweets', 4.2, 'Vadodara'),
    (6, 'Little Italy', 'Italian', 4.6, 'Ahmedabad'),
    (7, 'Mainland China', 'Chinese', 4.0, 'Ahmedabad'),
    (8, 'Punjab Grill', 'North Indian', 4.5, 'Surat'),
    (9, 'Sankalp', 'South Indian', 4.2, 'Surat'),
    (10, 'The Chocolate Room', 'Desserts', 4.3, 'Ahmedabad'),
    (11, 'Zaffran', 'Mughlai', 4.1, 'Vadodara'),
    (12, 'Agashiye', 'Gujarati Heritage', 4.8, 'Ahmedabad')
]
cursor.executemany("INSERT INTO restaurants VALUES (?, ?, ?, ?, ?);", restaurants_data)
conn.commit()

query_t4 = """
SELECT * 
FROM restaurants
ORDER BY name ASC
LIMIT 10;
"""

df_t4 = pd.read_sql_query(query_t4, conn)
print("Task 4 Output: First 10 Restaurants Alphabetically (Zomato A-Z Listing)")
display(df_t4)


Task 4 Output: First 10 Restaurants Alphabetically (Zomato A-Z Listing)


,id,name,cuisine,rating,city
0,12,Agashiye,Gujarati Heritage,4.8,Ahmedabad
1,2,Barbeque Nation,Buffet / BBQ,4.4,Ahmedabad
2,3,Cafe De Bistro,Continental,3.9,Surat
3,4,Domino's Pizza,Italian / Fast Food,4.1,Ahmedabad
4,5,Haldiram's,North Indian / Sweets,4.2,Vadodara
5,6,Little Italy,Italian,4.6,Ahmedabad
6,7,Mainland China,Chinese,4.0,Ahmedabad
7,8,Punjab Grill,North Indian,4.5,Surat
8,9,Sankalp,South Indian,4.2,Surat
9,1,Swagat Restaurant,South Indian,4.3,Ahmedabad


## Task 5: Top 3 Trending Songs with Tie-Breaker (`play_count DESC, added_at DESC`)
> **Problem Statement:** Suppose you want to display the top 3 trending songs from a `'songs'` table based on `play_count`, but if two songs have the same `play_count`, the more recently added song should come first. Write the SQL query to achieve this.  
> *Hint: Use `ORDER BY` with multiple columns.*

### SQL Query:
```sql
SELECT * 
FROM songs
ORDER BY play_count DESC, added_at DESC
LIMIT 3;
```

### Tie-Breaker Logic Explained:
* In our dataset, both `'Blinding Lights'` and `'Brown Munde'` have **identical play counts** ($2,100,000$).
* `'Blinding Lights'` was added on **`2026-10-22`**, whereas `'Brown Munde'` was added on **`2026-10-18`**.
* Because `added_at DESC` acts as the secondary tie-breaker, `'Blinding Lights'` (the more recent song) is awarded higher rank (#1), with `'Brown Munde'` taking rank #2.


In [5]:
# Create songs table
cursor.execute("DROP TABLE IF EXISTS songs;")
cursor.execute("""
CREATE TABLE songs (
    id INTEGER PRIMARY KEY,
    title VARCHAR(255) NOT NULL,
    artist VARCHAR(255) NOT NULL,
    play_count INTEGER NOT NULL,
    added_at TIMESTAMP NOT NULL
);
""")

songs_data = [
    (1, 'Starboy', 'The Weeknd', 1500000, '2026-10-15 10:00:00'),
    (2, 'Brown Munde', 'AP Dhillon', 2100000, '2026-10-18 12:00:00'),
    (3, 'Blinding Lights', 'The Weeknd', 2100000, '2026-10-22 14:30:00'),
    (4, 'Levitating', 'Dua Lipa', 1800000, '2026-10-20 09:15:00'),
    (5, 'Tum Hi Ho', 'Arijit Singh', 1950000, '2026-10-19 16:45:00')
]
cursor.executemany("INSERT INTO songs VALUES (?, ?, ?, ?, ?);", songs_data)
conn.commit()

query_t5 = """
SELECT * 
FROM songs
ORDER BY play_count DESC, added_at DESC
LIMIT 3;
"""

df_t5 = pd.read_sql_query(query_t5, conn)
print("Task 5 Output: Top 3 Trending Songs with Tie-Breaker (added_at DESC)")
display(df_t5)

# Clean up connection
conn.close()


Task 5 Output: Top 3 Trending Songs with Tie-Breaker (added_at DESC)


,id,title,artist,play_count,added_at
0,3,Blinding Lights,The Weeknd,2100000,2026-10-22 14:30:00
1,2,Brown Munde,AP Dhillon,2100000,2026-10-18 12:00:00
2,5,Tum Hi Ho,Arijit Singh,1950000,2026-10-19 16:45:00


---

## 📌 Summary of Sorting Techniques Mastered

| Requirement | SQL Pattern | Practical Scenario |
| :--- | :--- | :--- |
| **Low to High** | `ORDER BY price ASC` | Price filter on Flipkart / Amazon |
| **High to Low** | `ORDER BY price DESC` | Premium / luxury item search |
| **Top-N Slicing** | `ORDER BY col DESC LIMIT N` | "Top 5" or "Top 10" leaderboards |
| **Alphabetical A-Z** | `ORDER BY name ASC` | Directory listing (Zomato / Contacts) |
| **Multi-Tier Sorting** | `ORDER BY col1 DESC, col2 DESC` | Release date with rating tie-breaker |
| **Deterministic Tie-Breaking** | `ORDER BY metric DESC, timestamp DESC` | Trending algorithms (Spotify / YouTube) |
